# CellTypist vs CyteType agreement with cxg labels

Upstream steps (CellTypist, clustering, CyteType, CyteOnto) are run by [`run_pipeline.py`](run_pipeline.py). This notebook loads the cached annotated h5ad and CyteOnto CSV, then compares cxg (`cell_type`) and CyteType against CellTypist `predicted_labels` as the CyteOnto author reference.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

from cyteonto import attach_cytescores_to_obs
from cytetype_runner.utils import confidence_by_cluster
from shared.repo import REPO_ROOT

## Config
Get metadata for dataset to select the appropriate CellTypist model

In [ ]:
from metadata import sample_row_for_srx
from metadata.config import MetadataConfig

SRX = "SRX12708356"

cfg = MetadataConfig(
    sampleParquetPath=REPO_ROOT / "data/scbasecount/2026-01-12/metadata/GeneFull/Homo_sapiens/scbasecount_2026-01-12_metadata_GeneFull_Homo_sapiens_sample_metadata.parquet",
    obsParquetPath=REPO_ROOT / "data/scbasecount/2026-01-12/metadata/GeneFull/Homo_sapiens/scbasecount_2026-01-12_metadata_GeneFull_Homo_sapiens_obs_metadata.parquet",
)

sample = sample_row_for_srx(SRX, cfg)
sample

In [ ]:
from celltypist import models
models.models_description()

In [ ]:
CELLTYPIST_COL = "predicted_labels"
CXG_COL = "cell_type"
CYTETYPE_COL = "cytetype_annotation_leiden_merged"
AUTHOR_COL = CELLTYPIST_COL

OUTPUT_ROOT = REPO_ROOT / "output" / "celltypist_vs_cxg"
DATA_DIR = OUTPUT_ROOT / "data"
FIGS_DIR = REPO_ROOT / "writeups" / "celltypist_vs_cxg" / ".figs"

annotated_path = DATA_DIR / f"{SRX}_cytetype_annotated.h5ad"
cyteonto_csv_path = OUTPUT_ROOT / "cyteonto_results" / f"{SRX}_cyteonto.csv"

FIGS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
missing = [p for p in (annotated_path, cyteonto_csv_path) if not p.is_file()]
if missing:
    missing_str = "\n".join(f"  - {p}" for p in missing)
    raise FileNotFoundError(
        f"Cached outputs not found. Run run_pipeline.py first:\n"
        f"  uv run python writeups/celltypist_vs_cxg/run_pipeline.py --srx {SRX}\n"
        f"Missing:\n{missing_str}"
    )

adata = sc.read_h5ad(annotated_path)
cyteonto_df = pd.read_csv(cyteonto_csv_path)
cyteonto_df.head()

## Analysis: cxg vs cytetype agreement with CellTypist

In [ ]:
obs = adata.obs.copy()
obs = attach_cytescores_to_obs(
    obs,
    cyteonto_df,
    author_col=AUTHOR_COL,
    algorithm_col=CXG_COL,
    algorithm="cxg",
    out_col="cytescore_cxg",
)
obs = attach_cytescores_to_obs(
    obs,
    cyteonto_df,
    author_col=AUTHOR_COL,
    algorithm_col=CYTETYPE_COL,
    algorithm="cytetype",
    out_col="cytescore_cytetype",
)
obs = obs.dropna(subset=["cytescore_cxg", "cytescore_cytetype"])
obs[[AUTHOR_COL, CXG_COL, CYTETYPE_COL, "cytescore_cxg", "cytescore_cytetype"]].head()

In [ ]:
summary = (
    obs.groupby(AUTHOR_COL, observed=True)
    .agg(
        n_cells=(AUTHOR_COL, "size"),
        mean_cytescore_cxg=("cytescore_cxg", "mean"),
        mean_cytescore_cytetype=("cytescore_cytetype", "mean"),
    )
    .assign(
        delta_cxg_minus_cytetype=lambda df: (
            df["mean_cytescore_cxg"] - df["mean_cytescore_cytetype"]
        )
    )
    .sort_values("n_cells", ascending=False)
)
summary

In [ ]:
long = obs.melt(
    id_vars=[AUTHOR_COL],
    value_vars=["cytescore_cxg", "cytescore_cytetype"],
    var_name="method",
    value_name="cytescore_similarity",
)
long["method"] = long["method"].map(
    {
        "cytescore_cxg": "cxg",
        "cytescore_cytetype": "cytetype",
    }
)
order = summary.index.tolist()

fig, ax = plt.subplots(figsize=(8, max(4, len(order) * 0.25)))
sns.boxplot(
    data=long,
    x="cytescore_similarity",
    y=AUTHOR_COL,
    hue="method",
    order=order,
    fliersize=0,
    ax=ax,
)
ax.set_xlabel("CyteScore vs CellTypist author label")
ax.set_ylabel("CellTypist label")
ax.set_title(f"{SRX}: cytescore by method and CellTypist label")
fig.tight_layout()
fig.savefig(FIGS_DIR / f"cytescore_by_method_{SRX}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
delta = summary["delta_cxg_minus_cytetype"].sort_values()

fig, ax = plt.subplots(figsize=(7, max(4, len(delta) * 0.25)))
colors = ["#228B22" if v > 0 else "#d73027" for v in delta]
ax.barh(delta.index, delta.values, color=colors)
ax.axvline(0, color="grey", linewidth=0.5)
ax.set_xlabel("mean cytescore(cxg) - mean cytescore(cytetype)")
ax.set_ylabel("CellTypist label")
ax.set_title(f"{SRX}: which method agrees more with CellTypist per CellTypist label")
fig.tight_layout()
fig.savefig(FIGS_DIR / f"cytescore_delta_{SRX}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
confidence_map = confidence_by_cluster(adata)
obs["cytetype_confidence"] = obs["leiden_merged"].astype(str).map(confidence_map)

conf_order = ["Low", "Moderate", "High"]
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(
    data=obs,
    x="cytetype_confidence",
    y="cytescore_cytetype",
    order=[c for c in conf_order if c in obs["cytetype_confidence"].unique()],
    ax=ax,
)
ax.set_xlabel("CyteType cluster confidence")
ax.set_ylabel("CyteScore vs CellTypist (CyteType)")
ax.set_title(f"{SRX}: CyteType confidence vs CellTypist agreement")
fig.tight_layout()
fig.savefig(FIGS_DIR / f"cytescore_vs_confidence_{SRX}.png", dpi=150, bbox_inches="tight")
plt.show()